# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

In [1]:
# Task 1 - Các thành phần của bài toán tìm kiếm cho Dots and Boxes\n# State: Tập các đường đã nối (lines) và chủ sở hữu các ô (boxes); thông tin lượt chơi hiện tại.\n# Action: Chọn một đường (edge) chưa được nối; kết quả là cập nhật lines và có thể hoàn thành 1+ ô.\n# Transition: Từ state s, thực hiện action a -> state s' (nối đường); nếu action tạo ô thì người chơi được đi tiếp.\n# Initial state: Bàn trống (không có đường nào).\n# Terminal state: Khi tất cả các đường đã được nối (tức là không còn action nào).\n# Utility/Reward: Số ô của mỗi người khi game kết thúc (zero-sum được chuyển thành điểm cho người hiện tại).\ncomponents = {\n    'state_repr': 'set of filled edges and box owners',\n    'actions': 'available edges to draw',\n    'transition': 'place edge, possibly claim boxes, switch player unless box claimed',\n    'initial_state': 'empty board',\n    'terminal_test': 'no available edges',\n    'utility': 'difference in boxes (player1 - player2) or tuple of counts'\n}\nprint('Components defined. Ví dụ components dict:') \ncomponents\n

How big is the state space? Give an estimate and explain it.

In [2]:
# Ước lượng kích thước không gian trạng thái\ndef count_edges(rows, cols):\n    # rows x cols dots\n    hor = rows * (cols - 1)\n    ver = (rows - 1) * cols\n    return hor + ver\n\ndef estimate_state_space(rows, cols):\n    L = count_edges(rows, cols)\n    # Mỗi đường có thể tồn tại hoặc không -> 2^L trạng thái khả dĩ (không tất cả hợp lệ do turn order, nhưng là upper bound)\n    return L, 2**L\n\nL, space = estimate_state_space(4,4)\nprint(f'Board 4x4 dots -> edges L = {L}; upper-bound trạng thái = 2^L = {space} (≈ {space:.2e})')\n

How big is the game tree that minimax search will go through? Give an estimate and explain it.

In [3]:
# Ước lượng kích thước cây trò chơi cho minimax\ndef estimate_game_tree_upperbound(rows, cols):\n    L = count_edges(rows, cols)\n    # upper bound: mỗi lượt giảm 1 edge -> số lá ≤ L!\n    # Thực tế: nếu hoàn thành ô sẽ được đi tiếp, nhưng trên tổng thể số chiều sâu ≤ L\n    import math\n    return L, math.factorial(L)\n\nL, tree_upper = estimate_game_tree_upperbound(4,4)\nprint(f'L = {L}; Upper bound lá ≈ L! = {L}! = {tree_upper} (rất lớn, không thể duyệt đầy đủ)')\n

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [4]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

In [5]:
# Task 2 - Cài đặt lớp Board cho Dots and Boxes\nfrom collections import defaultdict\n\nclass Board:\n    def __init__(self, rows=4, cols=4):\n        # rows x cols điểm (dots)\n        self.rows = rows\n        self.cols = cols\n        # edges lưu dưới dạng tuple ((r,c), orientation) orientation 'h' hoặc 'v'\n        # nhưng để đơn giản ta biểu diễn edge bởi ((r1,c1),(r2,c2)) theo hai đỉnh\n        self.edges = set()  # các edge đã nối\n        # boxes: key = top-left coord of box, value = owner (None / 1 / -1)\n        self.boxes = dict(((r,c), None) for r in range(rows-1) for c in range(cols-1))\n        # tổng số edge\n        self.total_edges = self._compute_total_edges()\n    def _compute_total_edges(self):\n        return self.rows*(self.cols-1) + (self.rows-1)*self.cols\n    def all_edges(self):\n        edges = []\n        for r in range(self.rows):\n            for c in range(self.cols-1):\n                edges.append(((r,c),(r,c+1)))\n        for r in range(self.rows-1):\n            for c in range(self.cols):\n                edges.append(((r,c),(r+1,c)))\n        return edges\n    def available_actions(self):\n        return [e for e in self.all_edges() if e not in self.edges]\n    def play_edge(self, edge, player):\n        # edge: tuple of two dot coords\n        if edge in self.edges:\n            raise ValueError('Edge already played')\n        self.edges.add(edge)\n        completed = self._check_completed_boxes_by_edge(edge, player)\n        return completed  # số ô vừa được hoàn thành; caller sẽ quyết định có được đi tiếp hay không\n    def _check_completed_boxes_by_edge(self, edge, player):\n        # kiểm tra các ô xung quanh edge, nếu ô có 4 cạnh -> gán owner\n        completed = 0\n        # for each box top-left (r,c) check if edge touches it\n        for (r,c) in list(self.boxes.keys()):\n            if self.boxes[(r,c)] is not None:\n                continue\n            # các 4 edges của box\n            box_edges = {\n                ((r,c),(r,c+1)),\n                ((r,c+1),(r+1,c+1)),\n                ((r+1,c),(r+1,c+1)),\n                ((r,c),(r+1,c))\n            }\n            # normalize orientation (some edges may be reversed in representation)\n            norm_box_edges = set(tuple(sorted(e)) for e in box_edges)\n            # check if our stored edges use sorted tuples too\n            norm_played = set(tuple(sorted(e)) for e in self.edges)\n            if norm_box_edges.issubset(norm_played):\n                self.boxes[(r,c)] = player\n                completed += 1\n        return completed\n    def is_terminal(self):\n        return len(self.edges) == self.total_edges\n    def score(self):\n        # trả về (player1_score, player2_score) giả sử players 1 and -1\n        s1 = sum(1 for v in self.boxes.values() if v == 1)\n        s2 = sum(1 for v in self.boxes.values() if v == -1)\n        return s1, s2\n    def display(self):\n        # Hiển thị ASCII đơn giản: dot = o; horizontal = --- ; vertical = | ; empty = spaces\n        rows = self.rows\n        cols = self.cols\n        # build a grid of characters\n        for r in range(rows):\n            # dots and horizontal edges row\n            line = []\n            for c in range(cols):\n                line.append('o')\n                if c < cols-1:\n                    e = ((r,c),(r,c+1))\n                    if e in self.edges or tuple(sorted(e)) in self.edges:\n                        line.append('---')\n                    else:\n                        line.append('   ')\n            print(''.join(line))\n            # vertical edges and box owner row (except after last row)\n            if r < rows-1:\n                line = []\n                for c in range(cols):\n                    e = ((r,c),(r+1,c))\n                    if e in self.edges or tuple(sorted(e)) in self.edges:\n                        line.append('|')\n                    else:\n                        line.append(' ')\n                    if c < cols-1:\n                        owner = self.boxes[(r,c)]\n                        if owner == 1:\n                            line.append(' 1 ')\n                        elif owner == -1:\n                            line.append(' -1')\n                        else:\n                            line.append('   ')\n                print(''.join(line))\n

Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

In [6]:
# Random Agent: chọn action ngẫu nhiên trong available actions\nimport random\ndef random_agent(board, player):\n    acts = board.available_actions()\n    return random.choice(acts) if acts else None\n

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [7]:
# Ví dụ hiển thị board và chơi vài nước ngẫu nhiên\nb = Board(4,4)\nprint('Available edges:', len(b.available_actions()))\n# chơi 4 nước ngẫu nhiên để minh hoạ\nfor i in range(4):\n    a = random_agent(b, 1)\n    completed = b.play_edge(a, 1)\nprint('\nBoard sau 4 nước ngẫu nhiên (ví dụ):')\nb.display()\nprint('Boxes ownership:', b.boxes)\n

Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [8]:
# Minimax with Alpha-Beta pruning (depth-limited)\nimport math, time\n_nodes_expanded = 0\n\ndef heuristic(board, player):\n    # heuristic đơn giản: số ô hiện có của player minus đối thủ, plus potential (số box có 3 cạnh)\n    s1, s2 = board.score()\n    my = s1 if player==1 else s2\n    opp = s2 if player==1 else s1\n    # potential: boxes with 3 edges are risky (opponent can take); count boxes with 3 edges\n    played = set(tuple(sorted(e)) for e in board.edges)\n    potential = 0\n    for (r,c), owner in board.boxes.items():\n        if owner is None:\n            edges = [\n                tuple(sorted(((r,c),(r,c+1)))),\n                tuple(sorted(((r,c+1),(r+1,c+1)))),\n                tuple(sorted(((r+1,c),(r+1,c+1)))),\n                tuple(sorted(((r,c),(r+1,c)))),\n            ]\n            cnt = sum(1 for e in edges if e in played)\n            if cnt == 3:\n                potential -= 1  # boxes with 3 edges are dangerous\n            elif cnt == 2:\n                potential += 0.1\n    return (my - opp) + potential\n\ndef minimax(board, player, depth, alpha=-math.inf, beta=math.inf):\n    # returns (value, action) for player (current player to move)\n    global _nodes_expanded\n    _nodes_expanded = 0\n    def max_value(b, p, d, a, bta):\n        nonlocal _nodes_expanded\n        _nodes_expanded += 1\n        if d==0 or b.is_terminal():\n            return heuristic(b, player), None\n        v = -math.inf\n        best_act = None\n        for act in b.available_actions():\n            # copy board\n            nb = copy_board(b)\n            completed = nb.play_edge(act, p)\n            if completed>0:\n                # same player moves again\n                val,_ = max_value(nb, p, d-1, a, bta)\n            else:\n                val,_ = min_value(nb, -p, d-1, a, bta)\n            if val > v:\n                v = val; best_act = act\n            a = max(a, v)\n            if a >= bta:\n                break\n        return v, best_act\n    def min_value(b, p, d, a, bta):\n        nonlocal _nodes_expanded\n        _nodes_expanded += 1\n        if d==0 or b.is_terminal():\n            return heuristic(b, player), None\n        v = math.inf\n        best_act = None\n        for act in b.available_actions():\n            nb = copy_board(b)\n            completed = nb.play_edge(act, p)\n            if completed>0:\n                val,_ = min_value(nb, p, d-1, a, bta)\n            else:\n                val,_ = max_value(nb, -p, d-1, a, bta)\n            if val < v:\n                v = val; best_act = act\n            bta = min(bta, v)\n            if a >= bta:\n                break\n        return v, best_act\n    return max_value(board, player, depth, alpha, beta)\n

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

In [9]:
# Hàm copy nhanh cho Board (shallow copy + set copies)\ndef copy_board(board):\n    nb = Board(board.rows, board.cols)\n    nb.edges = set(tuple(sorted(e)) for e in board.edges)\n    nb.boxes = dict(board.boxes)\n    return nb\n

Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [10]:
# Ví dụ sử dụng minimax (depth-limited) chọn nước đi tốt nhất cho player 1\nb = Board(4,4)\n# chơi vài nước ngẫu nhiên để tạo vị trí giữa game\nfor _ in range(6):\n    a = random_agent(b, 1)\n    b.play_edge(a, 1)\nprint('Board trước khi minimax quyết định:'); b.display()\nval, act = minimax(b, 1, depth=4)\nprint('Minimax gợi ý action:', act, 'value:', val, 'nodes expanded:', _nodes_expanded)\n

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [11]:
def minimax_agent(board, player, depth=4):\n    v_act = minimax(board, player, depth)\n    return v_act[1]\n

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

In [12]:
# Hàm mô phỏng trận đấu giữa hai agent functions (agent(board,player) -> action)\ndef play_game(rows, cols, agent1, agent2, verbose=False, depth1=4, depth2=4):\n    b = Board(rows, cols)\n    player = 1\n    agents = {1: (agent1, depth1), -1: (agent2, depth2)}\n    while not b.is_terminal():\n        agent_func, d = agents[player]\n        if agent_func == minimax_agent:\n            act = agent_func(b, player, depth=d)\n        else:\n            act = agent_func(b, player)\n        if act is None:\n            break\n        completed = b.play_edge(act, player)\n        if verbose:\n            print(f'Player {player} plays {act}, completed {completed} boxes')\n            b.display(); print()\n        if completed==0:\n            player = -player  # đổi lượt nếu không hoàn thành ô\n    s1,s2 = b.score()\n    return s1, s2\n\n# Ví dụ: minimax vs random\ns = play_game(3,3, minimax_agent, random_agent, verbose=False, depth1=3)\nprint('Kết quả ví dụ (3x3 dots):', s)\n

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

In [13]:
# Chơi nhiều trận để so sánh\ndef evaluate_agents(n_games=10, rows=3, cols=3, depth=3):\n    import random\n    results = {'minimax_wins':0, 'draws':0, 'random_wins':0}\n    for i in range(n_games):\n        # alternate who is player 1 to reduce bias\n        if i % 2 == 0:\n            s1,s2 = play_game(rows, cols, minimax_agent, random_agent, depth1=depth)\n            score = s1 - s2\n            if score>0: results['minimax_wins']+=1\n            elif score==0: results['draws']+=1\n            else: results['random_wins']+=1\n        else:\n            s1,s2 = play_game(rows, cols, random_agent, minimax_agent, depth2=depth)\n            # if minimax is player -1 its score is s2\n            score = s2 - s1\n            if score>0: results['minimax_wins']+=1\n            elif score==0: results['draws']+=1\n            else: results['random_wins']+=1\n    return results\n\nprint('Đánh giá nhanh (10 games, 3x3 dots, depth=3):', evaluate_agents(10,3,3,3))\n

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [14]:
# Đo nodes expanded của minimax trên cùng một vị trí khi tăng depth\nb = Board(3,3)\nfor _ in range(4):\n    a = random_agent(b,1); b.play_edge(a,1)\nprint('Sample position:'); b.display()\nfor d in range(1,6):\n    global _nodes_expanded\n    _nodes_expanded = 0\n    _ = minimax(b, 1, depth=d)\n    print(f'depth={d} -> nodes expanded ~=', _nodes_expanded)\n

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

In [15]:
# Full match trace: two minimax agents (small board to keep computation low)\ns1,s2 = play_game(3,3, minimax_agent, minimax_agent, verbose=True, depth1=2, depth2=2)\nprint('Final score (minimax vs minimax):', (s1,s2))\n

### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [16]:
summary = """Bản giải đã thêm:\n- Định nghĩa bài toán tìm kiếm (state, action, transition, terminal, utility).\n- Lớp Board với phương thức available_actions, play_edge, display, score.\n- Random agent, Minimax (alpha-beta, depth-limited) và wrapper agent.\n- Hàm mô phỏng trận đấu và đánh giá nhanh giữa minimax và random.\n"""\nprint(summary)\n

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [17]:
print('Lý thuyết: Cây trò chơi có tối đa L! lá với L là số edges; đối với 4x4 dots, L=', count_edges(4,4))\n

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [18]:
# Helper: in ra các edge còn lại để debug/quan sát\ndef print_available_edges(board):\n    print('Remaining edges (count={}):'.format(len(board.available_actions())))\n    print(board.available_actions())\n

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

In [19]:
# Demo game and trace collection\nb = Board(3,3)\ntrace = []\nplayer = 1\nwhile not b.is_terminal():\n    act = random_agent(b, player)\n    completed = b.play_edge(act, player)\n    trace.append((player, act, completed))\n    if completed == 0:\n        player = -player\nprint('Trace sample (first 10 moves):', trace[:10])\n

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [20]:
print('Đã hoàn thành việc thay thế các chỗ chứa "# Your code/answer goes here." bằng lời giải và code minh hoạ.')\n